# ARIA → MintPy Time Series: Cecil Peak, Queenstown NZ

This notebook runs a Sentinel-1 InSAR time series for the Cecil Peak area using
pre-processed ARIA GUNW interferograms from ASF and MintPy small-baseline analysis.

**What you need before starting:**

1. A conda environment with ARIA-tools and MintPy installed:
   ```
   conda create -n aria_env python=3.10
   conda activate aria_env
   conda install -c conda-forge gdal shapely
   pip install ARIAtools mintpy
   ```
2. A free [NASA Earthdata](https://urs.earthdata.nasa.gov) account
3. The ASF bulk download script for your chosen track (see Section 1 below)
4. About **20 GB** of free disk space

---

**Study area:** Cecil Peak, Queenstown (~45.1°S, 168.6°E)  
**Tracks available:** Ascending Track 23, Ascending Track 96  
**Date range:** 2022–2024

## 1 · Download ARIA GUNW products from ASF

### Step 1a — Get the download script

1. Go to [ASF Vertex](https://search.asf.alaska.edu)
2. Set **Dataset → ARIA S1 GUNW**
3. Use this bounding box WKT in the search bar (or draw it on the map):
   ```
   POLYGON((168.5948 -45.1609,168.729 -45.1609,168.729 -45.0655,168.5948 -45.0655,168.5948 -45.1609))
   ```
4. Set **Filters → Track → 23** (or 96), **Date range → 2022-01-01 to 2024-12-31**
5. Click **Download All → Python script**
6. Save the `.py` script into your `gunw_products/` folder

### Step 1b — Run the download script

Open a terminal, activate your conda env, then:
```bash
cd cecilpeak_insar/gunw_products
python download-all-<date>.py
```
The script handles Earthdata authentication and will skip files already on disk if you re-run it.

In [ ]:
import os, glob

WORK_DIR   = './cecilpeak_insar'
GUNW_DIR   = os.path.join(WORK_DIR, 'gunw_products')
STACK_DIR  = os.path.join(WORK_DIR, 'stack')
MINTPY_DIR = os.path.join(WORK_DIR, 'mintpy')

for d in [GUNW_DIR, STACK_DIR, MINTPY_DIR]:
    os.makedirs(d, exist_ok=True)

print('Folders ready under', os.path.abspath(WORK_DIR))

Wait for the download to finish before continuing.  
The script will skip any files already on disk if you need to re-run it.

In [ ]:
# Check download is complete before continuing
nc_files = glob.glob(os.path.join(GUNW_DIR, '*.nc'))
print(f'Found {len(nc_files)} GUNW .nc files')
if len(nc_files) == 0:
    print('  --> Run the ASF download script first (see Step 1b above).')

## 2 · Prepare the stack with ARIA-tools

`ariaTSsetup.py` reads all the .nc files and builds the HDF5 stacks that MintPy expects.
This also downloads a DEM and tropospheric corrections — internet connection required.

Bounding box format is `S N W E`.

In [ ]:
BBOX = '-45.1609 -45.0655 168.5948 168.729'

cmd = (
    f'ariaTSsetup.py '
    f'-f "{GUNW_DIR}/*.nc" '
    f'--bbox "{BBOX}" '
    f'-w {STACK_DIR} '
    f'-nt 4'
)
print('Running:', cmd)
!{cmd}

## 3 · Write the MintPy template file

In [ ]:
template_path = os.path.join(MINTPY_DIR, 'template.txt')
stack_abs     = os.path.abspath(STACK_DIR)

template = f"""##–– MintPy template: Cecil Peak, Queenstown NZ (ARIA GUNW, Ascending)

mintpy.load.processor        = aria
mintpy.load.unwFile          = {stack_abs}/unwrapStack.vrt
mintpy.load.corFile          = {stack_abs}/cohStack.vrt
mintpy.load.connCompFile     = {stack_abs}/connCompStack.vrt
mintpy.load.demFile          = {stack_abs}/demStack.vrt
mintpy.load.incAngleFile     = {stack_abs}/incAngleStack.vrt
mintpy.load.azAngleFile      = {stack_abs}/azAngleStack.vrt
mintpy.load.waterMaskFile    = {stack_abs}/waterMaskStack.vrt

##–– Reference point (auto)
mintpy.reference.lalo        = auto

##–– Coherence threshold — lower = more pixels, higher = less noise
mintpy.networkInversion.minTempCoh = 0.7
"""

with open(template_path, 'w') as f:
    f.write(template)

print('Template written to', template_path)
print()
print(template)

## 4 · Run MintPy small-baseline analysis

In [ ]:
%%bash -s "$MINTPY_DIR" "$template_path"
cd "$1"
smallbaselineApp.py "$2"

## 5 · View your results

Run these in a terminal from the mintpy directory:

```bash
cd cecilpeak_insar/mintpy

# Mean velocity map
view.py velocity.h5

# Time series at a point — click on the map to select
tsview.py timeseries.h5
```

To zoom into Cecil Peak:
```bash
view.py velocity.h5 --sub-lat -45.17 -45.06 --sub-lon 168.59 168.73
```

---

**Sign convention reminder:** ARIA products have the opposite sign to HyP3 products.  
Positive LOS = motion **toward** the satellite; negative LOS = motion **away** from the satellite.